<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">K-Nearest Neighbours (KNN)</h1></center>

KNN is a **lazy learner** — it stores all training points and predicts by majority vote (classification) or mean (regression) of the k nearest neighbours.

### Topics:
1. [Theory — How KNN Works](#1)
2. [Dataset & Visualization](#2)
3. [Decision Boundary vs k](#3)
4. [Bias-Variance via k](#4)
5. [Evaluation of Model](#5)

In [ ]:
import seaborn as sns
knn_colors = ['#f72585', '#7209b7', '#3a0ca3', '#4361ee', '#4cc9f0']
print("KNN Colors:")
sns.palplot(sns.color_palette(knn_colors))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score,
                              precision_score, recall_score, f1_score)
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
np.random.seed(42)
print("Libraries loaded.")

## Let's Start!

<a id = "1"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Theory — How KNN Works</h1></center>

## Prediction

1. Compute the distance from the new point to **all** training points
2. Select the **k nearest** neighbours
3. Return the **majority class** (or mean for regression)

## Distance Weighting

- `weights="uniform"` — each neighbour gets equal vote
- `weights="distance"` — closer neighbours get weight $1/d$

## Bias-Variance Trade-off

| k | Bias | Variance | Boundary |
|---|---|---|---|
| Small (k=1) | Low | High | Very jagged |
| Large (k=n) | High | Low | Very smooth |

**Scaling is mandatory** — features with large ranges dominate distance.

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'],['\\(','\\)']]}});</SCRIPT>

<a id = "2"></a>
<center><h1 style = "background:#f72585 ;color:white;border:0;font-weight:bold">Information About Dataset</h1></center>

**Breast Cancer Wisconsin** — 569 samples, 30 features, binary target.

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
df = X.copy(); df["target"] = y
print(f"Shape: {df.shape}")
print(df["target"].value_counts().rename({0:"malignant",1:"benign"}))
df.describe().T.head(8)

<center><h1 style = "background:#7209b7 ;color:white;border:0;font-weight:bold">Data Visualization</h1></center>

In [ ]:
top_features = ["mean radius", "mean texture", "mean area", "mean concavity"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=100)

for ax, feat in zip(axes.ravel(), top_features):
    for label, color in [(0, "#f72585"), (1, "#4cc9f0")]:
        sns.kdeplot(df[df["target"]==label][feat], ax=ax,
                    label=data.target_names[label], color=color, fill=True, alpha=0.4)
    ax.set_title(feat); ax.legend()

plt.suptitle("Feature Distributions by Class", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(14, 8), dpi=100)
sns.heatmap(X.iloc[:, :10].assign(target=y).corr(numeric_only=True),
            annot=True, fmt=".2f", cmap="RdPu", linewidths=0.5)
plt.title("Correlation Heatmap (first 10 features)")
plt.show()

<center><h1 style = "background:#3a0ca3 ;color:white;border:0;font-weight:bold">Train-Test Split</h1></center>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Positive rate — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")

<a id = "3"></a>
<center><h1 style = "background:#4361ee ;color:white;border:0;font-weight:bold">Decision Boundary vs k</h1></center>

In [ ]:
X2d, y2d = make_moons(n_samples=400, noise=0.3, random_state=42)
X2d_tr, X2d_te, y2d_tr, y2d_te = train_test_split(X2d, y2d, test_size=0.3, random_state=42)

k_values = [1, 5, 20, 100]
fig, axes = plt.subplots(1, 4, figsize=(16, 4), dpi=100)

xx, yy = np.meshgrid(np.linspace(-2.5, 3.5, 300), np.linspace(-1.5, 2.5, 300))

for ax, k in zip(axes, k_values):
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    pipe.fit(X2d_tr, y2d_tr)
    Z   = pipe.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    acc = accuracy_score(y2d_te, pipe.predict(X2d_te))
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X2d_te[:,0], X2d_te[:,1], c=y2d_te, cmap="coolwarm",
               edgecolors="k", s=20)
    ax.set_title(f"k={k}\nTest acc={acc:.3f}", fontweight="bold")

plt.suptitle("KNN Decision Boundaries", fontweight="bold")
plt.tight_layout(); plt.show()

<a id = "4"></a>
<center><h1 style = "background:#4cc9f0 ;color:black;border:0;font-weight:bold">Bias-Variance via k</h1></center>

In [ ]:
k_range = range(1, 51)
train_err, test_err, cv_err = [], [], []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for k in k_range:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    pipe.fit(X_train, y_train)
    train_err.append(1 - accuracy_score(y_train, pipe.predict(X_train)))
    test_err.append(1  - accuracy_score(y_test,  pipe.predict(X_test)))
    cv = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="accuracy")
    cv_err.append(1 - cv.mean())

best_k = list(k_range)[np.argmin(cv_err)]

plt.figure(figsize=(11, 5), dpi=100)
plt.plot(list(k_range), train_err, "o-", ms=3, label="Train error", color="#f72585")
plt.plot(list(k_range), test_err,  "s-", ms=3, label="Test error",  color="#7209b7")
plt.plot(list(k_range), cv_err,    "d--", ms=3, label="CV error",   color="#4cc9f0")
plt.axvline(best_k, color="gray", ls=":", label=f"Best k={best_k}")
plt.xlabel("k"); plt.ylabel("Error Rate")
plt.title("KNN Bias-Variance Trade-off", fontweight="bold")
plt.legend(); plt.show()
print(f"Best k by CV: {best_k}")

<center><h1 style = "background:#7209b7 ;color:white;border:0;font-weight:bold">KNN Model</h1></center>

In [ ]:
best_pipe = make_pipeline(StandardScaler(),
                          KNeighborsClassifier(n_neighbors=best_k, weights="distance"))
best_pipe.fit(X_train, y_train)
pred = best_pipe.predict(X_test)
prob = best_pipe.predict_proba(X_test)[:, 1]

print(f"Train Accuracy: {best_pipe.score(X_train, y_train):.4f}")
print(f"Test  Accuracy: {accuracy_score(y_test, pred):.4f}")
print(f"ROC-AUC       : {roc_auc_score(y_test, prob):.4f}")

<a id = "5"></a>
<center><h1 style = "background:#3a0ca3 ;color:white;border:0;font-weight:bold">Evaluation of Model</h1></center>

In [ ]:
results = [accuracy_score(y_test, pred), precision_score(y_test, pred),
           recall_score(y_test, pred), f1_score(y_test, pred),
           roc_auc_score(y_test, prob)]
pd.DataFrame({"Metric": ["Accuracy","Precision","Recall","F1","ROC-AUC"],
              "Score": results}).round(4)

In [ ]:
print(classification_report(y_test, pred, target_names=data.target_names))

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5, 4), dpi=100)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title(f"Confusion Matrix — KNN (k={best_k})")
plt.show()